# Simple Neural Network (Multilayer Perceptron)

In [73]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import mlflow

In [74]:
train_df = pd.read_csv('../train_competition_2026.csv')
train_df.head()

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4,y_1,y_2
0,0,0,2068-09-19 23:34:11,1.38,49,7,1,3,1,0,1,105.5,95.0,67.4,36.6,23.2,33.4,107.4
1,0,0,2068-09-19 23:35:11,1.38,49,7,1,3,1,0,1,104.4,95.0,66.4,37.8,22.7,33.4,107.4
2,0,0,2068-09-19 23:36:11,1.38,49,7,1,3,1,0,1,104.0,95.0,65.2,37.0,22.1,33.4,107.4
3,0,0,2068-09-19 23:37:11,1.38,49,7,1,3,1,0,1,102.8,95.0,63.4,35.9,20.7,33.4,107.4
4,0,0,2068-09-19 23:38:11,1.38,49,7,1,3,1,0,1,101.3,95.1,59.1,34.5,18.1,33.4,107.4


In [75]:
train_df['time'] = pd.to_datetime(train_df['time'], format='%Y-%m-%d %H:%M:%S')

In [76]:
test_df = pd.read_csv('../test_no_outcome.csv')
test_df.head()

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4
0,18,1,2134-04-01 22:23:14,-1.0,38,1,1,1,0,0,0,105.4,99.8,50.7,61.4,36.8
1,18,1,2134-04-01 22:24:14,-1.0,38,1,1,1,0,0,0,105.4,99.4,49.4,61.1,36.2
2,18,1,2134-04-01 22:25:14,-1.0,38,1,1,1,0,0,0,104.6,99.0,49.7,61.4,36.6
3,18,1,2134-04-01 22:26:14,-1.0,38,1,1,1,0,0,0,104.5,99.6,51.7,61.8,37.2
4,18,1,2134-04-01 22:27:14,-1.0,38,1,1,1,0,0,0,104.6,99.5,52.5,61.9,37.5


## MLP with PCA

In [77]:
def flatten_blocks(df, feature_cols):
    X_flat = df.groupby(["obs"])[feature_cols].apply(lambda block: block.values.flatten())
    X_flat = pd.DataFrame(X_flat.tolist(), index=X_flat.index)
    return X_flat.reset_index()

In [78]:
feature_cols = [c for c in train_df.columns if c not in ["sub_id", "obs", "time", "y_1", "y_2"]]

X_train_flat = flatten_blocks(train_df, feature_cols)
X_test_flat  = flatten_blocks(test_df, feature_cols)

In [79]:
X_train_flat.head()

,obs,0,1,2,3,4,5,6,7,8,...,380,381,382,383,384,385,386,387,388,389
0,0,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,105.5,...,1.0,3.0,1.0,0.0,1.0,94.5,93.8,59.8,34.3,17.7
1,1,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,98.6,...,1.0,3.0,1.0,0.0,1.0,95.3,92.9,52.1,38.0,22.0
2,2,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,93.1,...,1.0,3.0,1.0,0.0,1.0,91.2,93.4,52.5,36.1,19.1
3,3,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,85.1,...,1.0,3.0,1.0,0.0,1.0,87.1,95.0,61.8,36.4,20.8
4,4,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,88.7,...,1.0,3.0,1.0,0.0,1.0,86.1,89.6,66.7,32.5,17.8


In [80]:
def extract_targets(df):
    return df.groupby(["obs"])[["y_1", "y_2"]].first().reset_index(drop=True)

y_train = extract_targets(train_df)

In [81]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers=1, hidden_size=32, dropout=0.0):
        super().__init__()
        
        layers = []
        
        # Input layer
        layers.append(nn.Linear(input_dim, hidden_size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        
        # Output layer
        layers.append(nn.Linear(hidden_size, output_dim))
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [82]:
def train_mlp(
    X_train, y_train, X_val, y_val,
    input_dim, output_dim, num_layers=1,
    hidden_size=32, dropout=0.0, weight_decay=1e-4,
    epochs=500, epoch_int=10, patience=20, lr=1e-3,
):
    
    model = SimpleMLP(
        input_dim=input_dim,
        output_dim=output_dim,
        num_layers=num_layers,
        hidden_size=hidden_size,
        dropout=dropout
    ).to("cpu")

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Training model with {param_count} params on {len(X_train)} samples: 1 to {len(X_train)/param_count} param to data ratio')
    criterion = nn.MSELoss()  # change for classification if needed
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32).to("cpu")
    y_train = torch.tensor(y_train, dtype=torch.float32).to("cpu")
    X_val   = torch.tensor(X_val, dtype=torch.float32).to("cpu")
    y_val   = torch.tensor(y_val, dtype=torch.float32).to("cpu")

    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()

        train_pred = model(X_train)
        train_loss = criterion(train_pred, y_train)

        train_loss.backward()
        optimizer.step()

        # Val
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val)
            
            # Calc r2
            train_r2 = r2_score(y_train.cpu().numpy(), train_pred.detach().cpu().numpy())
            val_r2 = r2_score(y_val.cpu().numpy(), val_pred.cpu().numpy())

        if epoch % epoch_int == 0:
            print(f'Epoch {epoch:03d} | '
                f'Train Loss: {train_loss.item():.4f} | Val Loss {val_loss.item():.4f} | '
                f'Train R2: {train_r2:.4f} | Val R2 {val_r2:.4f}')

        # Early stopping
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # Restore best model
    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        final_val_pred = model(X_val)
        final_val_loss = criterion(final_val_pred, y_val)
        print(f'Final val loss: {final_val_loss:.4f}')

    return model, final_val_loss

In [83]:
def run_pipeline(hp):
    k = hp['pca_k']
    
    scaler_x = StandardScaler()
    X_train_raw = X_train_flat.drop(columns=["obs"])
    
    X_train_scaled = scaler_x.fit_transform(X_train_raw)

    pca = PCA(n_components=k, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train)

    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train_pca, y_train_scaled, test_size=0.2, random_state=42
    )

    model, val_loss = train_mlp(
        X_train_split, y_train_split, X_val, y_val,
        input_dim=X_train_split.shape[1],
        output_dim=2,
        num_layers=hp['num_layers'],
        hidden_size=hp['hidden_size'],
        dropout=hp['dropout'],
        weight_decay=hp['weight_decay'],
        epochs=hp['epochs'],
        epoch_int=hp['epoch_int'],
        patience=hp['patience'],
        lr=hp['lr']
    )
    
    return model, val_loss, scaler_x, scaler_y, pca

In [84]:
hp = {
    'pca_k': 100,
    'hidden_size': 128,
    'num_layers': 4,
    'dropout': 0.0,
    'weight_decay': 1e-3,
    'epochs': 10000, 
    'epoch_int': 50,
    'patience': 50,
    'lr': 1e-3
}

model, val_loss, scaler_x, scaler_y, pca = run_pipeline(hp)

Training model with 62722 params on 11536 samples: 1 to 0.18392270654634738 param to data ratio
Epoch 000 | Train Loss: 1.0094 | Val Loss 0.9525 | Train R2: -0.0043 | Val R2 0.0273
Epoch 050 | Train Loss: 0.1949 | Val Loss 0.1984 | Train R2: 0.8058 | Val R2 0.7984
Epoch 100 | Train Loss: 0.1779 | Val Loss 0.1934 | Train R2: 0.8228 | Val R2 0.8035
Epoch 150 | Train Loss: 0.1616 | Val Loss 0.1965 | Train R2: 0.8390 | Val R2 0.8004
Early stopping at epoch 158
Final val loss: 0.1934


In [85]:
def get_real_predictions(model, X_test_df, scaler_x, scaler_y, pca):
    X_raw = X_test_df.drop(columns=["obs"]) if "obs" in X_test_df.columns else X_test_df
    X_scaled = scaler_x.transform(X_raw)
    
    X_pca = pca.transform(X_scaled)
    
    model.eval()
    X_tensor = torch.FloatTensor(X_pca)
    
    with torch.no_grad():
        y_pred_tensor = model(X_tensor)

    y_pred_scaled = y_pred_tensor.cpu().numpy()
    y_pred_final = scaler_y.inverse_transform(y_pred_scaled)
    
    return y_pred_final

In [86]:
predictions = get_real_predictions(model, X_test_flat, scaler_x, scaler_y, pca)

y_test_pred = get_real_predictions(model, X_test_flat, scaler_x, scaler_y, pca)

pred_df = pd.concat(
    [
        X_test_flat[["obs"]].reset_index(drop=True), 
        pd.DataFrame(y_test_pred, columns=["y_1", "y_2"])
    ],
    axis=1
)

len(y_test_pred)

3450

## MLP with Aggregated Features

In [105]:
def slope(values):
    x = np.arange(len(values)).reshape(-1, 1)
    return LinearRegression().fit(x, values).coef_[0]

def collapse_obs(df, feature_cols):
    
    def aggregate_group(group):
        group = group.sort_values("time")
        features = {}
        
        for col in feature_cols:
            vals = group[col].values
            
            features[f"{col}_min"] = np.min(vals)
            features[f"{col}_max"] = np.max(vals)
            features[f"{col}_mean"] = np.mean(vals)
            features[f"{col}_std"] = np.std(vals)
            
            features[f"{col}_first"] = vals[0]
            features[f"{col}_last"] = vals[-1]
            features[f"{col}_delta"] = vals[-1] - vals[0]
            features[f"{col}_slope"] = slope(vals)
        
        return pd.Series(features)

    X = (
        df
        .groupby("obs", group_keys=False)
        .apply(aggregate_group, include_groups=False)
        .reset_index()
    )
    
    return X

In [106]:
def collapse_target(df):
    y = (
        df
        .groupby(["obs"])[["y_1", "y_2"]]
        .first()
        .reset_index(drop=True)
    )
    return y

In [107]:
feature_cols = [
    c for c in train_df.columns
    if c not in ["obs", "sub_id", "time", "y_1", "y_2"]
]

X_train = collapse_obs(train_df, feature_cols).drop(columns=['obs'])
y_train = collapse_target(train_df)

X_test = collapse_obs(test_df, feature_cols)
test_index = X_test[["obs"]].copy()
X_test = X_test.drop(columns=['obs'])

In [108]:
X_train.head()

,num_0_min,num_0_max,num_0_mean,num_0_std,num_0_first,num_0_last,num_0_delta,num_0_slope,num_1_min,num_1_max,...,t_3_delta,t_3_slope,t_4_min,t_4_max,t_4_mean,t_4_std,t_4_first,t_4_last,t_4_delta,t_4_slope
0,1.38,1.38,1.38,4.440892e-16,1.38,1.38,0.0,4.611663e-33,49.0,49.0,...,-2.3,0.013415,14.3,28.8,21.880000,3.210234,23.2,17.7,-5.5,0.032392
1,1.38,1.38,1.38,4.440892e-16,1.38,1.38,0.0,4.611663e-33,49.0,49.0,...,0.3,-0.036129,19.8,30.3,24.026667,2.722001,19.8,22.0,2.2,0.011791
2,1.38,1.38,1.38,4.440892e-16,1.38,1.38,0.0,4.611663e-33,49.0,49.0,...,-3.3,-0.156196,17.5,29.0,21.453333,2.955417,24.4,19.1,-5.3,-0.259043
3,1.38,1.38,1.38,4.440892e-16,1.38,1.38,0.0,4.611663e-33,49.0,49.0,...,1.6,0.054683,17.4,27.8,20.696667,1.896046,19.3,20.8,1.5,0.078109
4,1.38,1.38,1.38,4.440892e-16,1.38,1.38,0.0,4.611663e-33,49.0,49.0,...,-3.8,-0.051880,17.8,25.0,21.500000,1.444761,21.8,17.8,-4.0,-0.014194


In [109]:
def extract_targets(df):
    return df.groupby(["obs"])[["y_1", "y_2"]].first().reset_index(drop=True)

y_train = extract_targets(train_df)

In [110]:
def run_pipeline(hp):
    scaler_x = StandardScaler()
    
    X_train_scaled = scaler_x.fit_transform(X_train)
    
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train)

    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train_scaled, y_train_scaled, test_size=0.2, random_state=42
    )

    model, val_loss = train_mlp(
        X_train_split, y_train_split, X_val, y_val,
        input_dim=X_train_split.shape[1],
        output_dim=2,
        num_layers=hp['num_layers'],
        hidden_size=hp['hidden_size'],
        dropout=hp['dropout'],
        weight_decay=hp['weight_decay'],
        epochs=hp['epochs'],
        epoch_int=hp['epoch_int'],
        patience=hp['patience'],
        lr=hp['lr']
    )
    
    return model, val_loss, scaler_x, scaler_y

In [114]:
hp = {
    'hidden_size': 256,
    'num_layers': 2,
    'dropout': 0.4,
    'weight_decay': 1e-3,
    'epochs': 10000, 
    'epoch_int': 50,
    'patience': 50,
    'lr': 5e-3
}

model, val_loss, scaler_x, scaler_y = run_pipeline(hp)

Training model with 93186 params on 11536 samples: 1 to 0.12379541991286244 param to data ratio
Epoch 000 | Train Loss: 1.0143 | Val Loss 0.5776 | Train R2: -0.0092 | Val R2 0.4095
Epoch 050 | Train Loss: 0.2169 | Val Loss 0.1929 | Train R2: 0.7839 | Val R2 0.8040
Epoch 100 | Train Loss: 0.2055 | Val Loss 0.1902 | Train R2: 0.7953 | Val R2 0.8068
Epoch 150 | Train Loss: 0.1999 | Val Loss 0.1908 | Train R2: 0.8008 | Val R2 0.8062
Early stopping at epoch 154
Final val loss: 0.1898


In [115]:
def get_real_predictions(model, X_test_df, scaler_x, scaler_y):
    X_raw = X_test_df.drop(columns=["obs"]) if "obs" in X_test_df.columns else X_test_df
    X_scaled = scaler_x.transform(X_raw)
    
    model.eval()
    X_tensor = torch.FloatTensor(X_scaled)
    
    with torch.no_grad():
        y_pred_tensor = model(X_tensor)

    y_pred_scaled = y_pred_tensor.cpu().numpy()
    y_pred_final = scaler_y.inverse_transform(y_pred_scaled)
    
    return y_pred_final

In [116]:
predictions = get_real_predictions(model, X_test, scaler_x, scaler_y)

y_test_pred = get_real_predictions(model, X_test, scaler_x, scaler_y)

pred_df = pd.concat(
    [
        X_test_flat[["obs"]].reset_index(drop=True), 
        pd.DataFrame(y_test_pred, columns=["y_1", "y_2"])
    ],
    axis=1
)

len(y_test_pred)

3450

## Saving and MLFlow

In [118]:
pred_df.to_csv("mlp_agg_1.csv", index=False)

In [113]:
def run_with_mlflow(hp):
    run_name = f"lstm_h{hp['hidden_size']}_l{hp['num_layers']}_d{hp['dropout']}_w{hp['weight_decay']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("grid_search", "agg_stats_mlp")
        mlflow.log_params(hp)
        model, val_loss, _, _ = run_pipeline(hp)
        mlflow.log_metric("val_loss", val_loss)
        
        return model, val_loss

if __name__ == '__main__':
    db_path = "/Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/mlflow.db"
    mlflow.set_tracking_uri(f"sqlite:///{db_path}")
    mlflow.set_experiment("AML_Project_MLP_Grid_Search")
    
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"Active Experiment: {mlflow.get_experiment_by_name('AML_Project_MLP_Grid_Search').experiment_id}")

    from itertools import product

    best_score = float("inf")
    
    hidden_sizes = [32, 64, 128, 256]
    num_layers = [2, 4, 6, 8]
    dropouts = [0.0, 0.2, 0.4]
    weight_decays = [1e-5, 1e-4, 1e-3]
    
    best_model = None
    # best_pca = None

    for hs, nl, dr, wd in product(hidden_sizes, num_layers, dropouts, weight_decays):
        
        hp = {
            'hidden_size': hs,
            'num_layers': nl,
            'dropout': dr,
            'weight_decay': wd,
            'epochs': 10000, 
            'epoch_int': 50,
            'patience': 50,
            'lr': 5e-3
        }
        
        model, score = run_with_mlflow(hp)
        
        if score < best_score:
            best_score = score
            best_hp = hp
            best_model = model
            
            # best_pca = PCA(n_components=hp['pca_k'], random_state=42)
            # best_pca.fit(X_train_flat.drop(columns=["obs"]))
            

    print(f"Best config: {best_hp}, val_score: {best_score}")

Tracking URI: sqlite:////Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/mlflow.db
Active Experiment: 1
Training model with 4482 params on 11536 samples: 1 to 2.5738509593931282 param to data ratio
Epoch 000 | Train Loss: 0.9992 | Val Loss 0.9083 | Train R2: 0.0058 | Val R2 0.0720
Epoch 050 | Train Loss: 0.2043 | Val Loss 0.2027 | Train R2: 0.7965 | Val R2 0.7940
Epoch 100 | Train Loss: 0.1916 | Val Loss 0.1958 | Train R2: 0.8091 | Val R2 0.8010
Epoch 150 | Train Loss: 0.1839 | Val Loss 0.1960 | Train R2: 0.8168 | Val R2 0.8009
Early stopping at epoch 180
Final val loss: 0.1955
Training model with 4482 params on 11536 samples: 1 to 2.5738509593931282 param to data ratio
Epoch 000 | Train Loss: 1.0096 | Val Loss 0.9186 | Train R2: -0.0044 | Val R2 0.0615
Epoch 050 | Train Loss: 0.2057 | Val Loss 0.2032 | Train R2: 0.7950 | Val R2 0.7935
Epoch 100 | Train Loss: 0.1920 | Val Loss 0.1967 | Train R2: 0.8087 | Val R2 0.8002
Epoch 150 | Train Loss: 0.1846 | Val Loss 0.1973 | Train R2: 0.81